In [1]:
from sqlalchemy.orm.attributes import get_state_history
%load_ext autoreload
%autoreload 2
"""1. Run pipeline to record checkpoints into SQLite database."""

from dotenv import load_dotenv
load_dotenv(override=True)

from data.sqlite_db import get_sqlite_checkpointer
from graphs.code_review_parentgraph import build_code_review_graph

# Build graph with SqliteSaver:
graph = build_code_review_graph(checkpointer=get_sqlite_checkpointer())

# Code for review:
code_snippet = """
def is_positive(num: int) -> bool:
    if num > 0:
        return True
    return False
"""

config = {"configurable": {"thread_id": "cli_review_session_05"}}

print("Running pipeline to generate checkpoints...")
result = graph.invoke(
    {
        "code": code_snippet,
        "language": "python",
        "file_path": None,
    },
    config=config,
)
print("Pipeline run finished! Checkpoints recorded into SQLite.")

Running pipeline to generate checkpoints...


D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(


Pipeline run finished! Checkpoints recorded into SQLite.


In [2]:
"""2. Inspect chronological checkpoint history recorded in SQLite database."""

from data.sqlite_db import get_sqlite_checkpointer
from graphs.code_review_parentgraph import build_code_review_graph
from utils.time_travel import inspect_state_history

graph = build_code_review_graph(checkpointer=get_sqlite_checkpointer())

# Retrieve full state timeline for this thread:
history = inspect_state_history(graph, thread_id="cli_review_session_05")

print(f"Total Checkpoints in DB: {len(history)}\n")

for item in history:
    step_num = item["step"]
    node_name = item["node"]
    next_nodes = item["next_nodes"]
    chk_id = item["checkpoint_id"]
    keys = item["state_keys"]

    print(f"• Step {step_num}: Node [{node_name}] -> Next: {next_nodes}")
    print(f"  Checkpoint ID: {chk_id}")
    print(f"  State Keys: {keys}\n")

Deserializing unregistered type output_schemas.code_review.ReviewResult from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('output_schemas.code_review', 'ReviewResult')]
Deserializing unregistered type output_schemas.code_review.TestSuite from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('output_schemas.code_review', 'TestSuite')]
Deserializing unregistered type output_schemas.code_review.TestExecutionResult from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('output_schemas.code_review', 'TestExecutionResult')]


Total Checkpoints in DB: 4

• Step -1: Node [input] -> Next: ['__start__']
  Checkpoint ID: 1f1a6fc3-7945-644c-bfff-ebd4e9b84b2c
  State Keys: []

• Step 0: Node [loop] -> Next: ['reviewer']
  Checkpoint ID: 1f1a6fc3-7954-66cf-8000-26a08a3856a5
  State Keys: ['code', 'language', 'file_path']

• Step 1: Node [loop] -> Next: ['test_generator']
  Checkpoint ID: 1f1a6fc3-9a3a-6039-8001-85b72b46712e
  State Keys: ['code', 'language', 'file_path', 'review', 'error']

• Step 2: Node [loop] -> Next: ['refactor_gate']
  Checkpoint ID: 1f1a6fc3-b440-682a-8002-5adefd249add
  State Keys: ['code', 'language', 'file_path', 'review', 'tests', 'test_execution', 'error']



In [1]:
"""Test python_interpreter tool invocation and state
persistence."""

from tools.interpreter import python_interpreter

print("=== Test 1: Assignment & State Creation ===")
res1 = python_interpreter.invoke(
    
    {
        "code": """
        users = [
        {"name": "Alice", "age": 28, "salary": 95000},
        {"name": "Bob", "age": 34, "salary": 110000},
        {"name": "Charlie", "age": 22, "salary": 62000},
    ]
print(f"Loaded {len(users)} records into memory.")
"""
    }
)
print(res1)

print("\n=== Test 2: State Persistence & Auto-Evaluation ===")
# Note: 'users' exists from Test 1 without being redefined!
res2 = python_interpreter.invoke(
    {
        "code": """
avg_salary = sum(u["salary"] for u in users) / len(users)
avg_salary
"""
    }
)
print(f"Calculated average salary: {res2}")

print("\n=== Test 3: Self-Correction Error Feedback ===")
res3 = python_interpreter.invoke({"code": "users[10]['salary']"})
print(f"Error output returned to agent:\n{res3}")

=== Test 1: Assignment & State Creation ===
Loaded 3 records into memory.


=== Test 2: State Persistence & Auto-Evaluation ===
Calculated average salary: 89000.0


=== Test 3: Self-Correction Error Feedback ===
Error output returned to agent:
Traceback (most recent call last):
  File "D:\Repos\lang_claude_exercise\core\src\tools\interpreter.py", line 56, in run
    result = eval(eval_code, self.globals)
  File "<repl>", line 1, in <module>
IndexError: list index out of range



In [ ]:
# """Stage 3: Multi-turn interactive test of the autonomous interpreter_agent."""
# Code-e Test baraye Notebook

# import time
# from agents.interpreter import interpreter_agent
# from dotenv import load_dotenv
# from langchain_core.messages import HumanMessage

# load_dotenv(override=True)

# # Shared thread ID to test cross-turn persistence in SQLite
# config = {"configurable": {"thread_id": "interpreter_interactive_demo_01"}}

# # --- Turn 1: Complex Prime & Square Calculation ---
# print("=== Turn 1: Calculating Sum of Squares of First 10 Primes ===")
# t0 = time.time()
# query1 = (
#     "Calculate the sum of squares of the first 10 prime numbers. Show your work"
#     " step by step."
# )

# res1 = interpreter_agent.invoke(
#     {"messages": [HumanMessage(content=query1)]},
#     config=config,
# )
# print(f"Turn 1 completed in {time.time()-t0:.2f}s!\n")
# print(res1["messages"][-1].content)


# # --- Turn 2: Follow-up calculation referencing Turn 1 state ---
# print("\n=== Turn 2: Follow-up Division on Persisted Thread ===")
# t1 = time.time()
# query2 = "Now divide that total sum by 10 using the interpreter."

# res2 = interpreter_agent.invoke(
#     {"messages": [HumanMessage(content=query2)]},
#     config=config,
# )
# print(f"Turn 2 completed in {time.time()-t1:.2f}s!\n")
# print(res2["messages"][-1].content)

In [1]:
from dotenv import load_dotenv
import time
from agents.interpreter import interpreter_agent
from langchain_core.messages import HumanMessage
from utils import get_text

In [2]:
query1 = (
        "Calculate the sum of squares of the first 10 prime numbers. Show your work"
        " step by step."
    )

query2 = "Now divide that total sum by 10 using the interpreter."

In [4]:
config = {"configurable": {"thread_id": "interpreter_interactive_demo_01"}}

res1 = interpreter_agent.invoke({"messages": [HumanMessage(content=query1)]}, config=config)

res2 = interpreter_agent.invoke({"messages": [HumanMessage(content=query2)]}, config=config)

print(get_text(res1["messages"][-1]), get_text(res2["messages"][-1]), sep="\n")

d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
d:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampli

Aval 10 ta adad-e aval ro peyda mikonim:
`2, 3, 5, 7, 11, 13, 17, 19, 23, 29`

Bad morabba'-e har kodoom ro mohasebe mikonim:
- $2^2 = 4$
- $3^2 = 9$
- $5^2 = 25$
- $7^2 = 49$
- $11^2 = 121$
- $13^2 = 169$
- $17^2 = 289$
- $19^2 = 361$
- $23^2 = 529$
- $29^2 = 841$

Dar nahayat in adad ro ba ham jam mikonim:
$4 + 9 + 25 + 49 + 121 + 169 + 289 + 361 + 529 + 841 = 2397$
H حاصل jam (2397) ro bar 10 taghsim kardim ke javab `239.7` shod.


In [36]:
names = []
for message in res1["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        for tc in message.tool_calls:
            names.append(tc["name"])

In [37]:
print(names)

['python_interpreter']
